# 05 · Headless & CI：没有界面也要能交付

**目标读者**：CI 维护者、批处理使用者

**核心目的**：把「问答」这件事从 Jupyter 里剥出来 —— 终端能答、离线能填、无人值守能降级，且产物可被独立校验。

**验收标准（8 条）**：

1. 无 widget 降级：`FINPASS_NO_WIDGET=1` 下 `render_question()` 返回 `str`；
2. `run --auto-only` 产出 5 件套（clean / changes / events / passport / diff）；
3. `resume` 批处理产出的 clean 与 02 的 clean **逐单元格相等**；
4. `verify` 三态：正常 0 / 有未解决 2 / 篡改 1；
5. 无头一致性：resume 与交互式在同一答案集下结果相同；
6. CI 复现：按护照里的 `cli_replay_cmd` 复跑，`output_file_sha256` 与原件一致；
7. `tests/test_cli_exit_codes.py` 3 passed；
8. `engine.py` 零金融词、未被改动。

**退出码三态（写死在 `cli.py`，notebook 只引用）**：

- `EXIT_OK = 0`：全部解决，无 flagged；
- `EXIT_UNRESOLVED = 2`：存在 flagged / `auto_only`。**软失败**——需要人工介入，但不该 fail build（对齐 grep / ruff 的惯例）；
- `EXIT_FAIL = 1`：文件缺失 / 哈希不符 / replay 失败 / 参数错。

> 合并成 0/1 会让「有未解决项」和「产物被篡改」变成同一个信号，CI 无法区分该通知谁。

## 大纲

1. 无 widget 降级：问题卡片变成纯文本
2. 批处理 run：`--auto-only` 与退出码 2
3. 批处理 resume：questions.yaml 回填，与 02 的 clean 逐单元格对齐
4. verify 三态：0 / 2 / 1
5. CI 复现：复现命令来自护照，不手拼

In [1]:
import json
import os
import shlex
import subprocess
import sys
from pathlib import Path

import pandas as pd
import yaml

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = REPO / "src"
sys.path.insert(0, str(SRC))

OUT02 = REPO / "out" / "02"
OUT05 = REPO / "out" / "05"
OUT05.mkdir(parents=True, exist_ok=True)
INPUT02 = OUT02 / "input_dirty.csv"
assert INPUT02.is_file(), f"缺 02 输入：{INPUT02}（05 不自己造脏数据）"

from finpass.session import open as open_session, render_question


def finpass(*args: str, stdin: str | None = None) -> subprocess.CompletedProcess:
    """统一封装：所有 CLI 调用都走真实进程，才能测到退出码。"""
    env = {**os.environ, "PYTHONPATH": str(SRC), "PYTHONIOENCODING": "utf-8"}
    return subprocess.run(
        [sys.executable, "-m", "finpass", *args],
        cwd=REPO, env=env, input=stdin, text=True, encoding="utf-8", capture_output=True, timeout=180,
    )


checks: dict[str, bool] = {}

os.environ["FINPASS_NO_WIDGET"] = "1"
session = open_session(INPUT02)
session.start()

rendered = []
for question in session.queue[:2]:
    card = render_question(question)
    rendered.append(card)
    print(card)
    print("-" * 60)

print("type =", type(rendered[0]).__name__)
assert all(isinstance(card, str) for card in rendered)
checks["C1 无 widget 降级返回 str"] = True

[duplicate_key] 层级=strategy 影响 12 行
默认决策：保留第一条
  1) 保留第一条
  2) 保留最后一条
  3) 跳过打标
  样本：{'日期': '2024-01-02'}
  样本：{'日期': '2024-01-03'}
  样本：{'日期': '2024-01-04'}
------------------------------------------------------------
[missing_token] 层级=column 影响 4 行
默认决策：填 NaN
  1) 填 NaN
  2) 跳过打标
  样本：{'日期': '2024-03-01', '收盘价': '--'}
  样本：{'日期': '2024-03-04', '收盘价': '--'}
  样本：{'日期': '2024-03-05', '收盘价': '--'}
------------------------------------------------------------
type = str


In [2]:
run_auto = OUT05 / "run_auto"
result = finpass("run", str(INPUT02), "--profile", "a_stock_daily", "--auto-only", "--out", str(run_auto))
print(result.stdout.strip().splitlines()[-1])
assert result.returncode == 2, result.stderr

artifact_names = {path.name for path in run_auto.iterdir()}
expected_suffixes = {".clean.csv", ".changes.parquet", ".events.jsonl", ".passport.json", ".diff.html"}
for suffix in expected_suffixes:
    assert any(name.endswith(suffix) for name in artifact_names), suffix

auto_passport = json.loads((run_auto / "input_dirty.passport.json").read_text(encoding="utf-8"))
print({key: auto_passport[key] for key in ("auto_only", "human_decisions_count", "unresolved_issues_count")})
assert auto_passport["auto_only"] is True
assert auto_passport["unresolved_issues_count"] > 0
assert auto_passport["human_decisions_count"] == 0
checks["C2 auto_only 5 件套 + rc=2"] = True

产物：d:\Files\project\finpass\out\05\run_auto | 退出码 2
{'auto_only': True, 'human_decisions_count': 0, 'unresolved_issues_count': 34}


In [3]:
# 答案集取自 02 的 events：逐 question_id 对应，保证「同答案」口径
events = [json.loads(line) for line in (OUT02 / "input_dirty.events.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()]
answers = {event["question_id"]: event for event in events}

questions_path = OUT05 / "questions_02.yaml"
draft = finpass("run", str(INPUT02), "--out", str(OUT05 / "draft"), "--questions", str(questions_path))
# 纯导出（未作答）没有产生任何 change，所以退出码是 0，不是 2
assert draft.returncode == 0, draft.stderr

payload = yaml.safe_load(questions_path.read_text(encoding="utf-8"))
assert {item["question_id"] for item in payload["questions"]} == set(answers), "问题集与 02 决策集不一致"
for item in payload["questions"]:
    event = answers[item["question_id"]]
    item["answer"] = {"action": event["action"], "params": event["params"], "reason": event["reason"]}
questions_path.write_text(yaml.safe_dump(payload, sort_keys=False, allow_unicode=True), encoding="utf-8")

resume_dir = OUT05 / "resume_filled"
resumed = finpass("resume", str(questions_path), "--out", str(resume_dir))
print(resumed.stdout.strip())
# 02 的决策集里有 flagged（ohlc 部分行 skip），按三态约定这是 2 而不是 0
assert resumed.returncode == 2, resumed.stderr

resumed_clean = pd.read_csv(resume_dir / "input_dirty.clean.csv")
reference_clean = pd.read_csv(OUT02 / "input_dirty.clean.csv")
# clean.csv 只有数据列（decision_id / ts 在 events 里），因此这里是纯数据严格相等
pd.testing.assert_frame_equal(resumed_clean, reference_clean)
print("resume clean 与 out/02 clean 严格相等：", resumed_clean.shape)
checks["C3 resume 与 02 clean 逐单元格一致"] = True

回填 5 题 | 产物：d:\Files\project\finpass\out\05\resume_filled | 退出码 2
resume clean 与 out/02 clean 严格相等： (42, 7)


In [4]:
normal_dir = OUT05 / "run_human"
normal = finpass(
    "run", str(INPUT02), "--out", str(normal_dir), "--interactive", "cli",
    stdin="1\n1\n1\n1\nb\n",
)
print("normal run rc =", normal.returncode)
assert normal.returncode == 0, normal.stderr

ok_verify = finpass("verify", str(normal_dir / "input_dirty.passport.json"))
unresolved_verify = finpass("verify", str(run_auto / "input_dirty.passport.json"))

tampered_dir = OUT05 / "tampered"
tampered_dir.mkdir(parents=True, exist_ok=True)
for path in normal_dir.iterdir():
    (tampered_dir / path.name).write_bytes(path.read_bytes())
tampered_clean = tampered_dir / "input_dirty.clean.csv"
text = tampered_clean.read_text(encoding="utf-8")
tampered_clean.write_text(text.replace("10.0", "99.9", 1), encoding="utf-8")
tampered_verify = finpass("verify", str(tampered_dir / "input_dirty.passport.json"))

codes = (ok_verify.returncode, unresolved_verify.returncode, tampered_verify.returncode)
print("verify 三态 rc =", codes)
assert codes == (0, 2, 1), codes
checks["C4 verify 三态 0/2/1"] = True

normal run rc = 0
verify 三态 rc = (0, 2, 1)


In [5]:
# C5 无头一致性：resume 与交互式在同一答案集下「过程等价」，而不只是「结果相等」
human_events = [
    json.loads(line)
    for line in (normal_dir / "input_dirty.events.jsonl").read_text(encoding="utf-8").splitlines()
    if line.strip()
]

# ---- 1) 反填：从交互轮的 events 重建答案集
same_path = OUT05 / "questions_same.yaml"
same_draft = finpass("run", str(INPUT02), "--out", str(OUT05 / "draft_same"), "--questions", str(same_path))
# 纯导出（未作答）不产生 change，退出码是 0
assert same_draft.returncode == 0, same_draft.stderr

draft_payload = yaml.safe_load(same_path.read_text(encoding="utf-8"))
by_question = {event["question_id"]: event for event in human_events}
assert {item["question_id"] for item in draft_payload["questions"]} == set(by_question), "问题集与交互式决策集不一致"
for item in draft_payload["questions"]:
    event = by_question[item["question_id"]]
    item["answer"] = {"action": event["action"], "params": event["params"], "reason": event["reason"]}
same_path.write_text(yaml.safe_dump(draft_payload, sort_keys=False, allow_unicode=True), encoding="utf-8")

# ---- 2) 反填无损：yaml 写回再读，问题必须与内存队列的「决策身份」逐字段一致
# samples / context_samples / estimated_changes 是展示字段，export_questions_yaml 不落盘；
# 判据因此是「哪个规则、哪一层级、哪些选项、影响哪些行」——这几项丢了，反填就是残缺的。
probe = open_session(INPUT02)
probe.start()
queue_by_id = {question.question_id: question for question in probe.queue}
roundtrip = yaml.safe_load(same_path.read_text(encoding="utf-8"))
assert set(queue_by_id) == {item["question_id"] for item in roundtrip["questions"]}, "反填后问题集与队列不等（丢问题或拆问题）"
for item in roundtrip["questions"]:
    origin = queue_by_id[item["question_id"]]
    assert item["rule_id"] == origin.rule_id, item["question_id"]
    assert item["level"] == origin.aggregation_level, item["question_id"]
    assert list(item["options"]) == list(origin.options), item["question_id"]
    assert item["default"] == origin.default, item["question_id"]
    assert [str(key) for key in item["affected_row_keys"]] == [str(key) for key in origin.affected_row_keys], item["question_id"]
    assert set(item["answer"]) == {"action", "params", "reason"}, item["question_id"]
print("反填无损：", len(roundtrip["questions"]), "个问题逐字段一致")

# ---- 3) 无头路径：用同一份答案集 resume
headless_dir = OUT05 / "headless_same"
same = finpass("resume", str(same_path), "--out", str(headless_dir))
print("resume(同答案集) rc =", same.returncode, "| 交互式 rc =", normal.returncode)
assert same.returncode == normal.returncode, same.stderr

# ---- 4) decision 序列逐条一致：clean 相等可能只是巧合，过程必须同构
def decisions(event_path):
    """按落盘顺序取出决策序列，只抹掉 decision_id 与 ts。"""
    rows = []
    for line in event_path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        event = json.loads(line)
        rows.append((
            event["rule_id"],
            event["action"],
            json.dumps(event["params"], sort_keys=True, ensure_ascii=False),
            tuple(str(key) for key in event["affected_row_keys"]),
        ))
    return rows

interactive_decisions = decisions(normal_dir / "input_dirty.events.jsonl")
headless_decisions = decisions(headless_dir / "input_dirty.events.jsonl")
assert len(interactive_decisions) == len(headless_decisions), (len(interactive_decisions), len(headless_decisions))
for index, (left, right) in enumerate(zip(interactive_decisions, headless_decisions), 1):
    assert left == right, f"第 {index} 条决策不一致：{left} != {right}"
print("decision 序列一致：", len(interactive_decisions), "条")

# ---- 5) changes 逐格一致：结果相同但改动机制不同，不算一致
def changes_table(out_dir):
    frame = pd.read_parquet(out_dir / "input_dirty.changes.parquet")
    frame = frame.drop(columns=["decision_id"])
    return frame.sort_values(["row_key", "col", "change_type"], kind="stable").reset_index(drop=True)

interactive_changes = changes_table(normal_dir)
headless_changes = changes_table(headless_dir)
assert len(interactive_changes) == len(headless_changes), (len(interactive_changes), len(headless_changes))
pd.testing.assert_frame_equal(interactive_changes, headless_changes)
print("changes 逐格一致：", len(interactive_changes), "条")

# ---- 6) 结果层：clean 逐单元格相等
pd.testing.assert_frame_equal(
    pd.read_csv(headless_dir / "input_dirty.clean.csv"),
    pd.read_csv(normal_dir / "input_dirty.clean.csv"),
)
print("clean 严格相等：", pd.read_csv(headless_dir / "input_dirty.clean.csv").shape)
checks["C5 无头一致性：resume 与交互式同答案集过程等价"] = True


反填无损： 5 个问题逐字段一致
resume(同答案集) rc = 0 | 交互式 rc = 0
decision 序列一致： 5 条
changes 逐格一致： 133 条
clean 严格相等： (42, 7)


In [6]:
# 复现命令必须来自护照，不许手拼
replay_cmd = auto_passport["cli_replay_cmd"]
print("cli_replay_cmd =", replay_cmd)
argv = shlex.split(replay_cmd)
assert argv[-2] == "--out", argv[-2:]

replay_dir = OUT05 / "replay_check"
argv[-1] = str(replay_dir)
# 未安装包时 `python -m finpass` 依赖 PYTHONPATH；复现命令本身只管命令，不管环境
replay_env = {**os.environ, "PYTHONPATH": str(SRC), "PYTHONIOENCODING": "utf-8"}
replay = subprocess.run(argv, cwd=REPO, env=replay_env, text=True, encoding="utf-8", capture_output=True, timeout=180)
print("replay rc =", replay.returncode)
assert replay.returncode == 2, replay.stderr

replayed_passport = json.loads((replay_dir / "input_dirty.passport.json").read_text(encoding="utf-8"))
# 判据只认文件字节 sha256：内存指纹已从护照移除，不做二次比较（口径见 passport.py 模块头）
for field in ("input_hash", "output_file_sha256", "profile_version"):
    print(field, auto_passport[field], "==", replayed_passport[field])
    assert auto_passport[field] == replayed_passport[field], field
checks["C6 复现命令来自护照且 output_file_sha256 一致"] = True

cli_replay_cmd = 'd:\Files\project\finpass\.venv\Scripts\python.exe' -m finpass run 'd:\Files\project\finpass\out\02\input_dirty.csv' --profile a_stock_daily --auto-only --out 'd:\Files\project\finpass\out\05\run_auto'
replay rc = 2
input_hash 2ae949fcaf0b6e93 == 2ae949fcaf0b6e93
output_file_sha256 c7a0d68d1f3f82001cac4fbae78beca90afb07a05b8994f8fa256ed89b2bd69b == c7a0d68d1f3f82001cac4fbae78beca90afb07a05b8994f8fa256ed89b2bd69b
profile_version 0.1.0 == 0.1.0


In [7]:
pytest_result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_cli_exit_codes.py", "-q"],
    cwd=REPO, text=True, encoding="utf-8", capture_output=True, timeout=300,
)
print(pytest_result.stdout.strip().splitlines()[-1])
assert pytest_result.returncode == 0, pytest_result.stdout
checks["C7 退出码单测 3 passed"] = True


3 passed in 13.35s


In [8]:
# C8 engine.py 未改动，且不含金融业务词（判据与守护测试同源）
import datetime
import hashlib

sys.path.insert(0, str(REPO / "tests"))
from test_no_finance_words_in_engine import scan_finance_words

engine_path = SRC / "finpass" / "engine.py"
baseline_path = REPO / "tests" / "baselines" / "engine.sha256"
baseline = baseline_path.read_text(encoding="ascii").strip()
digest = hashlib.sha256(engine_path.read_bytes()).hexdigest()
print("baseline =", baseline)
print("current  =", digest)
# 只补时点信息，不改判据：基线比引擎新 → 它是在引擎最后一次改动之后重生成的
print("baseline_mtime =", datetime.datetime.fromtimestamp(baseline_path.stat().st_mtime).isoformat(timespec="seconds"))
print("engine_mtime   =", datetime.datetime.fromtimestamp(engine_path.stat().st_mtime).isoformat(timespec="seconds"))
assert digest == baseline, "engine.py 与基线不一致：新增域只应新增 profiles/*.yaml"

hits = scan_finance_words(engine_path.read_text(encoding="utf-8"))
print("金融词命中 =", hits)
assert hits == [], hits
checks["C8 engine.py 未改动且零金融词"] = True


baseline = 8933beaae0084f20cc1c62970c52988e293007644f9594e8993081b11634a5ce
current  = 8933beaae0084f20cc1c62970c52988e293007644f9594e8993081b11634a5ce
baseline_mtime = 2026-11-02T16:47:49
engine_mtime   = 2026-11-02T16:28:38
金融词命中 = []


In [9]:

for name, passed in checks.items():
    print(f"{'✅' if passed else '❌'} {name}")
# 先查「条数够不够」，再查「有没有失败项」：少一条与某条挂掉，要能分开报
assert len(checks) == 8, sorted(checks)
assert all(checks.values())
print("✅ 05 验收通过 | 8/8 | 无头降级 · 三态退出码 · 复现命令来自护照 · 引擎零改动")


✅ C1 无 widget 降级返回 str
✅ C2 auto_only 5 件套 + rc=2
✅ C3 resume 与 02 clean 逐单元格一致
✅ C4 verify 三态 0/2/1
✅ C5 无头一致性：resume 与交互式同答案集过程等价
✅ C6 复现命令来自护照且 output_file_sha256 一致
✅ C7 退出码单测 3 passed
✅ C8 engine.py 未改动且零金融词
✅ 05 验收通过 | 8/8 | 无头降级 · 三态退出码 · 复现命令来自护照 · 引擎零改动


## 关于 outputs 的持久性

本 notebook 的 outputs 在 `Python 3.13.9 + pandas 3.1.0 + pyarrow 26.0.0` 下生成。
升级 pandas / pyarrow 后重跑，输出的数字可能变化，这**不一定**说明代码坏了：
复现以 `.venv` 重跑为准，判据是 cell 内的 `assert`，不是 output 文本。

`out/05/` 下的子目录是 C2–C6 的验收中间证据（清单见 `out/05/README.md`），
不作为交付产物、不参与下游消费；保留它们是为了让审计员能复核「clean 相等」是不是真比对过。
